# Day 30: Fetch Layer: an Entry Dataclass, httpx Timeouts and Retries, Paginated Fetch & MockTransport Tests (23 Nov 2026)

**Week 5 (consolidation week), end of Phase 1 (Foundation).** Day 2 of 7 in Week 5: the first layer of the thin-sync project.

**Goal:** pull every entry of a content type out of a (fake) headless CMS reliably: typed data, timeouts, safe retries, cursor pagination, and tests that never touch the network.

**Deliverables today:** `day30/fetch.py` (7 TODOs) with the 27 given tests in `test_fetch.py` green, `day30/test_mine.py` (3 tests of your own) and `day30/dsa.py` (dedupe_latest, longest_failure_streak).

The full lesson, task brief and answers are in the master Google Doc. This notebook holds the runnable examples. Run the cells top to bottom. You need `httpx` and `pytest` installed (`pip install httpx pytest`, or `uv add httpx pytest`). Concept 5 writes two files into a `scratch/` folder next to the notebook, which you can delete afterwards.

## Recap answers (fill in before running anything)

1. What is the difference between `json.loads()` and `json.load()`, and which exception does `json.loads()` raise on broken text (and what is its parent class)?
2. What does `@dataclass` write for you, and what does `frozen=True` add?
3. Day 22: a GET that returned 503, a POST without an Idempotency-Key that timed out, and a GET that returned 404. Which one is safe to retry automatically, and why?

*Your answers:*

## Concept 1: An Entry dataclass that parses at the edge

Raw API JSON is a dict of strings. Convert it **once**, at the boundary, into a typed object (`@dataclass(frozen=True)`) with a `from_api()` classmethod. Every later layer (Postgres on Day 31, FastAPI on Day 32) can then trust the data. Bad items raise **one** error type (`ValueError`) that names the item.

In React terms: this is the adapter you write between `fetch()` and your components, so components never see the raw API shape.

In [1]:
# Parse one raw CMS item (a dict) into a typed, frozen dataclass.
from dataclasses import dataclass
from datetime import datetime


# frozen=True: no field can change after creation (like Object.freeze in JS)
@dataclass(frozen=True)
class Article:
    uid: str
    title: str
    locale: str
    updated_at: datetime  # a real datetime, not the raw string
    # a tuple, not a list: a frozen object should hold immutable values
    tags: tuple[str, ...] = ()

    # classmethod: called on the class itself, Article.from_api(raw).
    # Like a static factory in JS: static fromApi(raw) { ... }
    @classmethod
    def from_api(cls, raw: dict) -> "Article":
        """Build an Article from the CMS's JSON shape, or raise ValueError."""
        try:
            return cls(
                uid=raw["uid"],  # required: KeyError if missing
                title=raw["title"].strip(),
                locale=raw.get("locale", "en-us"),  # optional, with default
                # fromisoformat accepts a trailing "Z" on Python 3.11+
                updated_at=datetime.fromisoformat(raw["updated_at"]),
                tags=tuple(raw.get("tags") or []),  # None or [] -> ()
            )
        except (KeyError, TypeError, ValueError) as exc:
            # REMEMBER: one clear error type out; "from exc" keeps the cause
            uid = raw.get("uid")
            raise ValueError(f"bad entry {uid!r}: {exc!r}") from exc


raw = {"uid": "blt01", "title": "  Hello  ",
       "updated_at": "2026-11-20T10:00:00Z", "tags": None}
a = Article.from_api(raw)
print(a.title, a.locale, a.tags)  # -> Hello en-us ()
print(a.updated_at.tzinfo)  # -> UTC  (the "Z" became a real timezone)
print(a == Article.from_api(raw))  # -> True  (dataclass writes __eq__)

try:
    Article.from_api({"uid": "blt02"})
except ValueError as e:
    print(e)  # -> bad entry 'blt02': KeyError('title')

Hello en-us ()
UTC
True
bad entry 'blt02': KeyError('title')


**Remember:** parse at the edge, once. One `from_api()` knows the CMS's JSON shape; everything after it works with typed, frozen objects. Timestamps carry a timezone.

## Concept 2: httpx.Client, timeouts and the two kinds of failure

`httpx` is the modern HTTP client (sync and async, same API). A `Client` keeps a **connection pool**, a `base_url`, default headers and a `Timeout`. Two different kinds of failure:

| Failure | What you get | Example |
|---|---|---|
| The server answered with an error | a `Response` with status 4xx/5xx (no exception until `raise_for_status()`) | 401, 404, 503 |
| No usable answer at all | an exception: `httpx.TransportError` (`TimeoutException`, `ConnectError` ...) | timeout, DNS failure |

`MockTransport(handler)` replaces the network with your own function, which is how every example and test today runs offline.

In [2]:
# One httpx.Client: base_url, default headers, a timeout, a fake server.
import httpx


def fake_cms(request: httpx.Request) -> httpx.Response:
    # MockTransport calls this for every request: no network involved.
    if request.headers.get("access_token") != "secret-token":
        return httpx.Response(401, json={"error": "bad token"})
    return httpx.Response(200, json={"path": request.url.path,
                                     "params": dict(request.url.params)})


client = httpx.Client(
    base_url="https://cms.example.test/v3",  # every path is joined to this
    headers={"access_token": "secret-token"},  # sent on every request
    # 10 s to read, write or wait for a pooled connection; 3 s to connect
    timeout=httpx.Timeout(10.0, connect=3.0),
    transport=httpx.MockTransport(fake_cms),  # swap in the fake server
)

# "with" closes the client's connection pool at the end, like Day 4's files
with client:
    # params builds the query string for you, like URLSearchParams in JS
    r = client.get("/content_types/blog/entries", params={"limit": 2})
    print(r.status_code)  # -> 200
    print(r.json())
    # -> {'path': '/v3/content_types/blog/entries', 'params': {'limit': '2'}}
    # GOTCHA: query values come back as strings ('2'), never ints

    # a per-request header overrides the client default
    bad = client.get("/anything", headers={"access_token": "wrong"})
    print(bad.status_code, bad.is_success)  # -> 401 False
    # GOTCHA: httpx does NOT raise on 4xx/5xx by itself, like fetch() in JS
    try:
        bad.raise_for_status()  # turns a 4xx/5xx into an exception
    except httpx.HTTPStatusError as exc:
        print("HTTPStatusError", exc.response.status_code)
        # -> HTTPStatusError 401

200
{'path': '/v3/content_types/blog/entries', 'params': {'limit': '2'}}
401 False
HTTPStatusError 401


In [3]:
# What a timeout looks like: an exception, not a response.
import httpx


def too_slow(request: httpx.Request) -> httpx.Response:
    # pretend the server never answered in time
    raise httpx.ReadTimeout("no reply in 10 s", request=request)


with httpx.Client(transport=httpx.MockTransport(too_slow)) as c:
    try:
        c.get("https://cms.example.test/v3/entries")
    except httpx.TimeoutException as exc:
        print(type(exc).__name__)  # -> ReadTimeout
        # REMEMBER: timeouts and connection errors are TransportErrors
        print(isinstance(exc, httpx.TransportError))  # -> True
        print(isinstance(exc, httpx.HTTPStatusError))  # -> False

ReadTimeout
True
False


**Remember:** httpx does not raise on 4xx/5xx by itself, and a timeout is an exception, not a response. Always set a timeout you chose (the httpx default is 5 s for every phase).

## Concept 3: Retries with exponential backoff

Retry only what can succeed next time: **transport errors, 429, 500, 502, 503, 504**. Never retry 400/401/403/404/422: the same request will fail the same way. Wait longer each time (0.5, 1, 2, 4 s, capped), honour `Retry-After` when the server sends it, add jitter when many clients retry together, and stop after a fixed number of attempts.

Pass `sleep` in as a parameter so tests can use a fake that records the waits instead of waiting.

In [4]:
# Retry a flaky call with exponential backoff, with a fake sleep.
import random


def delay_for(attempt: int, base: float = 0.5, cap: float = 8.0) -> float:
    # attempt 1 -> 0.5 s, 2 -> 1 s, 3 -> 2 s ... never more than cap
    return min(cap, base * 2 ** (attempt - 1))


print([delay_for(n) for n in range(1, 7)])
# -> [0.5, 1.0, 2.0, 4.0, 8.0, 8.0]  (the cap stops the growth)

# FDE TIP: add "jitter" (a random part) so 100 clients that failed together
# do not all retry at the same instant and knock the server over again
rng = random.Random(42)  # seeded only so this example prints the same
print(round(rng.uniform(0, delay_for(3)), 3))  # -> 1.279


class TemporaryError(Exception):
    """Stands in for a timeout or a 503."""


answers = [TemporaryError("503"), TemporaryError("timeout"), "ok"]


def flaky_call() -> str:
    # pops the next scripted answer: fails twice, then succeeds
    item = answers.pop(0)
    if isinstance(item, Exception):
        raise item
    return item


def call_with_retry(fn, max_attempts: int, sleep) -> str:
    # "sleep" is passed in, so a test can pass a fake that sleeps 0 s
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()  # success: leave the loop straight away
        except TemporaryError as exc:
            if attempt == max_attempts:
                raise  # out of attempts: let the caller see the error
            wait = delay_for(attempt)
            print(f"attempt {attempt} failed ({exc}); waiting {wait}s")
            sleep(wait)
    raise AssertionError("unreachable")  # keeps type checkers happy


slept = []  # a fake sleep that records instead of waiting
print(call_with_retry(flaky_call, max_attempts=4, sleep=slept.append))
# -> attempt 1 failed (503); waiting 0.5s
# -> attempt 2 failed (timeout); waiting 1.0s
# -> ok
print(slept)  # -> [0.5, 1.0]  (no real time passed)

[0.5, 1.0, 2.0, 4.0, 8.0, 8.0]
1.279
attempt 1 failed (503); waiting 0.5s
attempt 2 failed (timeout); waiting 1.0s
ok
[0.5, 1.0]


**Remember:** retry the temporary, fail fast on the permanent, cap the waits, and inject `sleep` so tests take milliseconds.

## Concept 4: Pagination and generators

APIs return data in pages. **Offset** pagination (`skip`/`limit`, as in Contentstack's Delivery API) is simple but can skip or repeat items when data changes while you page. **Cursor** pagination (`next_cursor`, used in today's task) asks the server "give me what comes after this point". Either way: loop until the server says there is no more, and guard against a server that never stops (`max_pages`, repeated cursors).

A generator (`yield`, Day 11) hands out items one at a time and fetches the next page only when asked.

In [5]:
# Offset pagination (skip/limit, the style Contentstack's Delivery API uses).
from collections.abc import Iterator

import httpx

ITEMS = [{"uid": f"blt{i:02}"} for i in range(1, 6)]  # 5 fake entries
calls = []  # records every request the fake server receives


def fake_cms(request: httpx.Request) -> httpx.Response:
    calls.append(str(request.url.params))
    skip = int(request.url.params.get("skip", 0))
    limit = int(request.url.params.get("limit", 100))
    page = ITEMS[skip: skip + limit]  # slicing past the end gives []
    return httpx.Response(200, json={"entries": page, "count": len(ITEMS)})


# Iterator[str]: "this function yields strings" (a generator, Day 11)
def iter_uids(client: httpx.Client, limit: int = 2) -> Iterator[str]:
    skip = 0
    while True:
        r = client.get("/entries", params={"skip": skip, "limit": limit})
        r.raise_for_status()
        data = r.json()
        batch = data["entries"]
        for item in batch:
            yield item["uid"]  # hand out one uid, pause here until asked
        skip += len(batch)
        # stop on an empty page OR when we have seen everything
        if not batch or skip >= data["count"]:
            return


client = httpx.Client(base_url="https://cms.example.test/v3",
                      transport=httpx.MockTransport(fake_cms))
with client:
    uids = iter_uids(client)  # GOTCHA: nothing is fetched yet (lazy)
    print(len(calls))  # -> 0
    print(next(uids), len(calls))  # -> blt01 1  (first page fetched)
    print(list(uids))  # -> ['blt02', 'blt03', 'blt04', 'blt05']
    print(calls)
    # -> ['skip=0&limit=2', 'skip=2&limit=2', 'skip=4&limit=2']

0
blt01 1
['blt02', 'blt03', 'blt04', 'blt05']
['skip=0&limit=2', 'skip=2&limit=2', 'skip=4&limit=2']


**Remember:** loop until the server says stop, but never trust it to stop: cap the pages and detect a repeated cursor. Generators are lazy: nothing is fetched until you iterate.

## Concept 5: Testing HTTP code with MockTransport and pytest

Give the code under test a client whose transport is a fake handler. The handler can return any status, raise a timeout, and **record each request** so the test can check the URL, params and headers too. A fixture that returns a factory lets each test build its own fake server. The next cell writes the module under test, the one after writes the tests, and the last one runs pytest.

In [6]:
# Create the scratch folder the next two cells write into.
from pathlib import Path

Path("scratch").mkdir(exist_ok=True)  # no error if it already exists

In [7]:
%%writefile scratch/latest.py
# The code under test: one small function that talks to the CMS.
import httpx


def latest_title(client: httpx.Client) -> str:
    """Return the title of the most recently updated entry, or ""."""
    r = client.get("/entries", params={"limit": 1, "order": "-updated_at"})
    r.raise_for_status()  # a 4xx/5xx becomes httpx.HTTPStatusError
    items = r.json()["entries"]
    return items[0]["title"] if items else ""

Writing scratch/latest.py


In [8]:
%%writefile scratch/test_latest.py
# Testing HTTP code with httpx.MockTransport and pytest (no network).
import httpx
import pytest

from latest import latest_title

BASE = "https://cms.example.test/v3"


# A fixture that returns a FUNCTION: each test builds its own fake server.
# Like a factory you would pass to a React test's render() wrapper.
@pytest.fixture
def make_client():
    def _make(handler) -> httpx.Client:
        return httpx.Client(base_url=BASE,
                            transport=httpx.MockTransport(handler))
    return _make


def test_sends_the_right_query(make_client):
    seen = []  # the handler stores each request so the test can inspect it

    def handler(request: httpx.Request) -> httpx.Response:
        seen.append(request)
        return httpx.Response(200, json={"entries": [{"title": "Home"}]})

    assert latest_title(make_client(handler)) == "Home"
    # REMEMBER: assert on the REQUEST too, not only on the result
    assert seen[0].url.path == "/v3/entries"
    assert seen[0].url.params["order"] == "-updated_at"


def test_empty_list_gives_empty_string(make_client):
    def handler(request):
        return httpx.Response(200, json={"entries": []})

    assert latest_title(make_client(handler)) == ""


# parametrize runs the same test once per status code (Day 10)
@pytest.mark.parametrize("status", [401, 404, 500, 503])
def test_error_status_raises(make_client, status):
    def handler(request):
        return httpx.Response(status, json={"error": "nope"})

    # pytest.raises passes only if the block raises this exception type
    with pytest.raises(httpx.HTTPStatusError):
        latest_title(make_client(handler))


def test_timeout_is_not_swallowed(make_client):
    def handler(request):
        raise httpx.ConnectTimeout("no route", request=request)

    with pytest.raises(httpx.TimeoutException):
        latest_title(make_client(handler))

Writing scratch/test_latest.py


In [9]:
# Run pytest on the scratch folder, as you would in a terminal.
import subprocess
import sys

# sys.executable = this notebook's Python, so the right packages are used
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no",
     "scratch"],
    capture_output=True, text=True,
)
print(result.stdout.strip().splitlines()[-1])  # -> 7 passed in 0.05s
# 7, not 4: parametrize turned one test into four cases (401 ... 503)

7 passed in 0.05s


**Remember:** test the unhappy paths (401, 503, timeouts, empty pages) with a fake transport, and assert on the requests you sent, not only on the result.

## Check your understanding (do NOT run until you have predicted)

Write your predictions here first, then paste the code into a new cell and run it.

```python
# Day 30 check: predict every numbered print, then run the file.
import dataclasses
from dataclasses import dataclass

import httpx


@dataclass(frozen=True)
class Entry:
    uid: str
    tags: tuple[str, ...] = ()


# 1) changing a frozen dataclass
e = Entry("blt01")
try:
    e.uid = "blt99"
except dataclasses.FrozenInstanceError:
    print(1, "frozen")
print(1, dataclasses.replace(e, uid="blt99"), e.uid)


def echo(request: httpx.Request) -> httpx.Response:
    # the fake server sends back the full URL it received
    return httpx.Response(200, text=str(request.url))


# 2) base_url joining, with and without a leading slash
c = httpx.Client(base_url="https://cms.test/v3",
                 transport=httpx.MockTransport(echo))
print(2, c.get("entries").text)
print(2, c.get("/entries").text)

# 3) a None value in params
print(3, c.get("/entries", params={"limit": 2, "cursor": None}).text)

# 4) status checks
r = httpx.Response(304)
print(4, r.is_success, r.is_redirect, r.is_error)


# 5) a generator that is never iterated
hits = []


def pages():
    hits.append("fetch")
    yield [1, 2]


g = pages()
print(5, len(hits))
```

*Your predictions:*

## Today's task

Open `fetch.py`, `test_mine.py` and `dsa.py` in this folder. Each has the brief, acceptance criteria and step-by-step hints; `mock_cms.py` is the fake CMS and `test_fetch.py` holds the 27 given test cases. Run `python -m pytest -q` inside `day30/` after each TODO. When done, push `day30/` and paste your work into the Forward Deployment Engineer project chat as **"Day 30 review"**.